<a href="https://colab.research.google.com/github/alxmzr/Colab/blob/main/%D0%A1ycle_Mirror.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- АДАВТИРАВАННЫЕ НАСТРАЙКИ ---
TICKER = "ETH-USD"
LOOKBACK = 60       # Дни для анализа
FORECAST = 45       # Прогноз
TOP_N = 3           # Количество лучших циклов

# Загрузка данных
df = yf.download(TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

current_df = df.iloc[-LOOKBACK:]
current_prices = current_df['Close'].values
current_dates = current_df.index

# Нормализация (Z-score) для поиска
target_norm = (current_prices - np.mean(current_prices)) / np.std(current_prices)

results = []
search_space = df.iloc[:-FORECAST]
for i in range(len(search_space) - LOOKBACK):
    sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
    sample_norm = (sample - np.mean(sample)) / np.std(sample)
    score = np.mean((target_norm - sample_norm)**2)
    results.append((score, i))

results.sort(key=lambda x: x[0])
best_cycles_data = []
used_indices = []
for score, idx in results:
    if not any(abs(idx - used_idx) < 30 for used_idx in used_indices):
        # Расчет корреляции Пирсона для этого участка
        hist_prices = search_space.iloc[idx : idx + LOOKBACK]['Close'].values
        correlation = np.corrcoef(current_prices, hist_prices)[0, 1]
        best_cycles_data.append((idx, correlation))
        used_indices.append(idx)
    if len(best_cycles_data) == TOP_N: break

# Подготовка данных для интервалов
forecast_matrix = []
future_dates = [current_dates[-1] + timedelta(days=i) for i in range(1, FORECAST + 1)]

# Визуализация
fig = go.Figure()
fig.add_trace(go.Scatter(x=current_dates, y=current_prices, name="ТЕКУЩАЯ ЦЕНА", line=dict(color='black', width=4)))

colors = ['red', 'blue', 'green']
for i, (idx, corr) in enumerate(best_cycles_data):
    raw_values = search_space.iloc[idx : idx + LOOKBACK + FORECAST]['Close'].values
    ratio = current_prices[-1] / raw_values[LOOKBACK-1]
    proj_values = raw_values[LOOKBACK:] * ratio
    forecast_matrix.append(proj_values)

    start_date = search_space.index[idx].strftime('%Y-%m-%d')
    fig.add_trace(go.Scatter(
        x=future_dates,
        y=proj_values,
        name=f"Цикл {i+1} ({start_date}) Corr: {corr:.2%}",
        line=dict(color=colors[i], width=2, dash='dot')
    ))

forecast_matrix = np.array(forecast_matrix)
mean_forecast = np.mean(forecast_matrix, axis=0)
upper_bound = np.max(forecast_matrix, axis=0)
lower_bound = np.min(forecast_matrix, axis=0)

# Доверительный интервал
fig.add_trace(go.Scatter(
    x=future_dates + future_dates[::-1],
    y=list(upper_bound) + list(lower_bound)[::-1],
    fill='toself', fillcolor='rgba(0,100,80,0.1)', line=dict(color='rgba(255,255,255,0)'),
    hoverinfo="skip", name="Диапазон (Top-3)"
))

# Средний прогноз
fig.add_trace(go.Scatter(x=future_dates, y=mean_forecast, name="СРЕДНИЙ ПРОГНОЗ", line=dict(color='orange', width=3)))

fig.update_layout(
    title=f"<b>Прогноз {TICKER} с мат. сходством (Pearson Correlation)</b>",
    xaxis_title="Дата", yaxis_title="Цена",
    template="plotly_white", hovermode="x unified"
)
if "USD" in TICKER: fig.update_yaxes(type="log")
fig.show()

[*********************100%***********************]  1 of 1 completed


In [2]:
# --- VISUALIZING MATCH QUALITY (ACTUAL VS MATCHES) ---
import plotly.graph_objects as go
import numpy as np

fig_match = go.Figure()

# 1. Current Actual Prices (Normalized for shape comparison)
current_prices_norm = (current_prices - np.mean(current_prices)) / np.std(current_prices)
fig_match.add_trace(go.Scatter(
    x=np.arange(LOOKBACK),
    y=current_prices_norm,
    name="Actual Price (Z-Score)",
    line=dict(color='black', width=4)
))

# 2. Top 3 Best Matching Historical Fractals (Normalized)
colors = ['red', 'blue', 'green']
for i, (idx, corr) in enumerate(best_cycles_data):
    hist_sample = df.iloc[idx : idx + LOOKBACK]['Close'].values
    hist_norm = (hist_sample - np.mean(hist_sample)) / np.std(hist_sample)
    start_date = df.index[idx].strftime('%Y-%m-%d')

    fig_match.add_trace(go.Scatter(
        x=np.arange(LOOKBACK),
        y=hist_norm,
        name=f"Match #{i+1} (Start: {start_date})",
        line=dict(color=colors[i], width=2, dash='dot'),
        opacity=0.7
    ))

fig_match.update_layout(
    title=f"<b>Fractal Fit Analysis: Actual vs. Historical Matches ({TICKER})</b><br><sup>Comparing normalized shapes over the {LOOKBACK}-day lookback period</sup>",
    xaxis_title="Days in Lookback Window",
    yaxis_title="Normalized Price (Z-Score)",
    template="plotly_white",
    hovermode="x unified"
)

fig_match.show()

In [3]:
# --- VISUALIZING MATCH QUALITY (ACTUAL VS MATCHES) ---
import plotly.graph_objects as go

fig_match = go.Figure()

# 1. Current Actual Prices (Normalized for shape comparison)
current_prices_norm = (current_prices - np.mean(current_prices)) / np.std(current_prices)
fig_match.add_trace(go.Scatter(
    x=np.arange(LOOKBACK),
    y=current_prices_norm,
    name="Actual Price (Z-Score)",
    line=dict(color='black', width=4)
))

# 2. Top 3 Best Matching Historical Fractals (Normalized)
colors = ['red', 'blue', 'green']
for i, (idx, corr) in enumerate(best_cycles_data):
    hist_sample = df.iloc[idx : idx + LOOKBACK]['Close'].values
    hist_norm = (hist_sample - np.mean(hist_sample)) / np.std(hist_sample)
    start_date = df.index[idx].strftime('%Y-%m-%d')

    fig_match.add_trace(go.Scatter(
        x=np.arange(LOOKBACK),
        y=hist_norm,
        name=f"Match #{i+1} (Start: {start_date})",
        line=dict(color=colors[i], width=2, dash='dot'),
        opacity=0.7
    ))

fig_match.update_layout(
    title=f"<b>Fractal Fit Analysis: Actual vs. Historical Matches ({TICKER})</b><br><sup>Comparing normalized shapes over the {LOOKBACK}-day lookback period</sup>",
    xaxis_title="Days in Lookback Window",
    yaxis_title="Normalized Price (Z-Score)",
    template="plotly_white",
    hovermode="x unified"
)

fig_match.show()

### Forecast Summary Table
This table presents the projected price levels for key dates in the future based on the Top-3 matching fractals.

In [4]:
# --- FORECAST TABLE GENERATION ---

# Create a DataFrame for the forecast
forecast_df = pd.DataFrame({
    'Date': [d.strftime('%Y-%m-%d') for d in future_dates],
    'Mean_Projected': mean_forecast,
    'Low_Scenario': lower_bound,
    'High_Scenario': upper_bound
})

# Filter for weekly milestones (every 7 days) and the final day
steps = list(range(0, len(forecast_df), 7))
if (len(forecast_df) - 1) not in steps:
    steps.append(len(forecast_df) - 1)

summary_table = forecast_df.iloc[steps].copy()

# Round for readability
summary_table = summary_table.round(2)

print(f"Forecast Summary for {TICKER} (Next {FORECAST} Days):")
display(summary_table)

Forecast Summary for ETH-USD (Next 45 Days):


,Date,Mean_Projected,Low_Scenario,High_Scenario
0,2026-05-11,2468.94,2399.76,2599.71
7,2026-05-18,2462.60,2337.07,2679.84
14,2026-05-25,2560.39,2402.20,2828.10
21,2026-06-01,2503.02,2086.55,2805.48
28,2026-06-08,2408.66,1934.21,2781.24
35,2026-06-15,2458.67,2011.25,2937.42
42,2026-06-22,2372.99,1995.77,2678.48
44,2026-06-24,2387.14,1919.05,2623.83


In [6]:
# --- VISUALIZING MATCH QUALITY (ACTUAL VS MATCHES) ---
import plotly.graph_objects as go
import numpy as np

fig_match = go.Figure()

# 1. Current Actual Prices (Normalized for shape comparison)
current_prices_norm = (current_prices - np.mean(current_prices)) / np.std(current_prices)
fig_match.add_trace(go.Scatter(
    x=np.arange(LOOKBACK),
    y=current_prices_norm,
    name="Actual Price (Z-Score)",
    line=dict(color='black', width=4)
))

# 2. Top 3 Best Matching Historical Fractals (Normalized)
colors = ['red', 'blue', 'green']
# Fixed: using best_cycles_data and unpacking (idx, corr)
for i, (idx, corr) in enumerate(best_cycles_data):
    hist_sample = df.iloc[idx : idx + LOOKBACK]['Close'].values
    hist_norm = (hist_sample - np.mean(hist_sample)) / np.std(hist_sample)
    start_date = df.index[idx].strftime('%Y-%m-%d')

    fig_match.add_trace(go.Scatter(
        x=np.arange(LOOKBACK),
        y=hist_norm,
        name=f"Match #{i+1} (Start: {start_date})",
        line=dict(color=colors[i], width=2, dash='dot'),
        opacity=0.7
    ))

fig_match.update_layout(
    title=f"<b>Fractal Fit Analysis: Actual vs. Historical Matches ({TICKER})</b><br><sup>Comparing normalized shapes over the {LOOKBACK}-day lookback period</sup>",
    xaxis_title="Days in Lookback Window",
    yaxis_title="Normalized Price (Z-Score)",
    template="plotly_white",
    hovermode="x unified"
)

fig_match.show()

In [10]:
# --- CONSOLIDATED COMPARISON ANALYSIS ---

# 1. Recalculate Backtest Top-3 (100 days ago)
BACKTEST_DAYS_AGO = 100
cutoff_idx = len(df) - BACKTEST_DAYS_AGO
test_data = df.iloc[:cutoff_idx]

target_series_bt = test_data['Close'].iloc[-LOOKBACK:].values
target_norm_bt = (target_series_bt - np.mean(target_series_bt)) / np.std(target_series_bt)

search_space_bt = test_data.iloc[:-FORECAST]
results_bt = []
for i in range(len(search_space_bt) - LOOKBACK):
    sample = search_space_bt.iloc[i : i + LOOKBACK]['Close'].values
    sample_norm = (sample - np.mean(sample)) / np.std(sample)
    score = np.mean((target_norm_bt - sample_norm)**2)
    results_bt.append((score, i))

results_bt.sort(key=lambda x: x[0])
backtest_best_indices = []
used_bt = []
for s, idx in results_bt:
    if not any(abs(idx - u) < 30 for u in used_bt):
        backtest_best_indices.append(idx)
        used_bt.append(idx)
    if len(backtest_best_indices) == TOP_N: break

# 2. Extract Current Model Data
current_best_indices = [item[0] for item in best_cycles_data]
current_best_dates = [df.index[idx].strftime('%Y-%m-%d') for idx in current_best_indices]
backtest_best_dates = [df.index[idx].strftime('%Y-%m-%d') for idx in backtest_best_indices]

# 3. Build Comparison Table
comparison_df = pd.DataFrame({
    'Rank': [f'#{i+1}' for i in range(len(current_best_dates))],
    'Current Model Cycle Start': current_best_dates,
    'Backtest (100d ago) Cycle Start': backtest_best_dates
})

print("Comparison of Top-3 Fractal Matches:")
display(comparison_df)

# Check for overlap
overlap = set(current_best_dates).intersection(set(backtest_best_dates))
if overlap:
    print(f"\nCommon cycles found in both models: {list(overlap)}")
else:
    print("\nNo exact overlapping cycles found. This is normal as the current model has 100 more days of data available.")

Comparison of Top-3 Fractal Matches:


,Rank,Current Model Cycle Start,Backtest (100d ago) Cycle Start
0,#1,2021-07-03,2022-09-21
1,#2,2019-04-15,2019-10-01
2,#3,2019-01-15,2018-11-22



No exact overlapping cycles found. This is normal as the current model has 100 more days of data available.


In [11]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- НАСТРОЙКИ БЭКТЕСТА ---
BACKTEST_DAYS_AGO = 100  # Точка в прошлом для проверки
LOOKBACK = 60
FORECAST = 60
TOP_N = 3

# Используем уже загруженные данные df
# Точка отсечки для теста
cutoff_idx = len(df) - BACKTEST_DAYS_AGO
test_data = df.iloc[:cutoff_idx]
actual_after = df.iloc[cutoff_idx : cutoff_idx + FORECAST]

# Текущий паттерн на момент отсечки
target_series = test_data['Close'].iloc[-LOOKBACK:].values
target_norm = (target_series - np.mean(target_series)) / np.std(target_series)
target_dates = test_data.index[-LOOKBACK:]

# Поиск по истории ДО точки отсечки
search_space = test_data.iloc[:-FORECAST]
results = []
for i in range(len(search_space) - LOOKBACK):
    sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
    sample_norm = (sample - np.mean(sample)) / np.std(sample)
    score = np.mean((target_norm - sample_norm)**2)
    results.append((score, i))

results.sort(key=lambda x: x[0])
best_indices = []
used = []
for s, idx in results:
    if not any(abs(idx - u) < 30 for u in used):
        best_indices.append(idx)
        used.append(idx)
    if len(best_indices) == TOP_N: break

# Сбор прогнозов
forecast_matrix = []
future_dates = [target_dates[-1] + timedelta(days=i) for i in range(1, FORECAST + 1)]

for idx in best_indices:
    raw = search_space.iloc[idx : idx + LOOKBACK + FORECAST]['Close'].values
    ratio = target_series[-1] / raw[LOOKBACK-1]
    forecast_matrix.append(raw[LOOKBACK:] * ratio)

forecast_matrix = np.array(forecast_matrix)
mean_f = np.mean(forecast_matrix, axis=0)

# --- ВИЗУАЛИЗАЦИЯ БЭКТЕСТА ---
fig = go.Figure()

# Реальные данные (до и после)
fig.add_trace(go.Scatter(x=target_dates, y=target_series, name="Факт (до теста)", line=dict(color='black', width=3)))
fig.add_trace(go.Scatter(x=actual_after.index, y=actual_after['Close'], name="Факт (реальное будущее)", line=dict(color='orange', width=3)))

# Прогноз
fig.add_trace(go.Scatter(x=future_dates, y=mean_f, name="Средний прогноз фракталов", line=dict(color='green', dash='dash')))

fig.update_layout(
    title="<b>Бэктест: Сравнение прогноза с реальными данными в прошлом</b>",
    xaxis_title="Дата", yaxis_title="Цена",
    template="plotly_white", hovermode="x unified"
)
fig.show()

In [12]:
import itertools

# --- ПАРАМЕТРЫ ДЛЯ ТЕСТИРОВАНИЯ ---
lookback_options = [30, 60, 90]
forecast_options = [30, 45, 60]
BACKTEST_DAYS = 100

results_grid = []

# Отрезаем данные для теста
cutoff_idx = len(df) - BACKTEST_DAYS
test_data = df.iloc[:cutoff_idx]
actual_future = df.iloc[cutoff_idx : cutoff_idx + max(forecast_options)]

print("Запуск оптимизации сетки...")

for lb, fc in itertools.product(lookback_options, forecast_options):
    # Текущий паттерн
    target = test_data['Close'].iloc[-lb:].values
    target_norm = (target - np.mean(target)) / np.std(target)

    # Поиск лучшего фрактала в истории
    best_score = float('inf')
    best_idx = -1
    search_space = test_data.iloc[:-fc]

    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb]['Close'].values
        sample_norm = (sample - np.mean(sample)) / np.std(sample)
        score = np.mean((target_norm - sample_norm)**2)
        if score < best_score:
            best_score = score
            best_idx = i

    # Проверка точности прогноза
    if best_idx != -1:
        raw_hist = search_space.iloc[best_idx : best_idx + lb + fc]['Close'].values
        ratio = target[-1] / raw_hist[lb-1]
        pred = raw_hist[lb:] * ratio

        # Считаем ошибку относительно реальности
        actual = df['Close'].iloc[cutoff_idx : cutoff_idx + fc].values
        rmse = np.sqrt(np.mean((pred - actual)**2))
        results_grid.append({'Lookback': lb, 'Forecast': fc, 'RMSE': rmse})

# Вывод результатов
optimization_df = pd.DataFrame(results_grid).sort_values('RMSE')
print("\nЛучшие параметры по итогам бэктеста:")
display(optimization_df.head())

# Визуализация матрицы ошибок
pivot_df = optimization_df.pivot(index='Lookback', columns='Forecast', values='RMSE')
fig = go.Figure(data=go.Heatmap(
    z=pivot_df.values,
    x=[f'Forecast {c}' for c in pivot_df.columns],
    y=[f'Lookback {r}' for r in pivot_df.index],
    colorscale='Viridis'))
fig.update_layout(title='Карта ошибок (RMSE) параметров фрактала')
fig.show()

Запуск оптимизации сетки...

Лучшие параметры по итогам бэктеста:


,Lookback,Forecast,RMSE
6,90,30,203.931475
7,90,45,431.003691
8,90,60,460.208130
0,30,30,678.834893
4,60,45,699.264495


In [13]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- НАСТРОЙКИ ---
TICKER = "BTC-USD"          # Можно заменить на ^GSPC (S&P500)
LOOKBACK_WINDOW = 360       # Длина отрезка для сравнения (например, последние 60 дней)
FORECAST_WINDOW = 30       # На сколько дней вперед мы хотим видеть прогноз из прошлого

# --- 1. ЗАГРУЗКА ДАННЫХ ---
df = yf.download(TICKER, start="2010-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Берем текущее движение (последние 60 дней)
current_move = df['Close'].iloc[-LOOKBACK_WINDOW:].values
current_move_norm = (current_move / current_move[0])

# --- 2. ПОИСК ПОХОЖЕГО ЦИКЛА (АЛГОРИТМ) ---
best_score = float('inf')
best_idx = -1

# Проходим по всей истории и ищем минимальную разницу (MSE)
for i in range(len(df) - LOOKBACK_WINDOW - FORECAST_WINDOW):
    past_move = df['Close'].iloc[i : i + LOOKBACK_WINDOW].values
    past_move_norm = (past_move / past_move[0])

    # Считаем разницу между текущим фракталом и историческим
    score = np.sum((current_move_norm - past_move_norm)**2)

    if score < best_score:
        best_score = score
        best_idx = i

# Извлекаем найденный цикл + его будущее продолжение
found_cycle = df.iloc[best_idx : best_idx + LOOKBACK_WINDOW + FORECAST_WINDOW]
found_date = found_cycle.index[0].strftime('%Y-%m-%d')
found_values_norm = (found_cycle['Close'].values / found_cycle['Close'].values[0]) * 100

# Текущие данные для графика (в % от старта окна)
current_plot = (current_move / current_move[0]) * 100

# --- 3. ВИЗУАЛИЗАЦИЯ ---
fig = go.Figure()

# Линия текущего движения
fig.add_trace(go.Scatter(
    x=np.arange(LOOKBACK_WINDOW),
    y=current_plot,
    name="ТЕКУЩЕЕ ДВИЖЕНИЕ",
    line=dict(color='black', width=4)
))

# Линия найденного исторического цикла
fig.add_trace(go.Scatter(
    x=np.arange(LOOKBACK_WINDOW + FORECAST_WINDOW),
    y=found_values_norm,
    name=f"НАЙДЕННЫЙ ЦИКЛ (от {found_date})",
    line=dict(color='red', width=2, dash='dot')
))

# Выделяем зону прогноза
fig.add_vrect(
    x0=LOOKBACK_WINDOW-1, x1=LOOKBACK_WINDOW + FORECAST_WINDOW - 1,
    fillcolor="green", opacity=0.1, layer="below", line_width=0,
    annotation_text="ВЕРОЯТНОЕ БУДУЩЕЕ"
)

fig.update_layout(
    title=f"Поиск фрактала для {TICKER} (Найден цикл от {found_date})",
    xaxis_title="Дни",
    yaxis_title="Изменение цены (%)",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()


[*********************100%***********************]  1 of 1 completed


In [19]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime, timedelta

# --- 1. НАСТРОЙКИ ---
TICKER = "BTC-USD"           # Инструмент (S&P 500)
LOOKBACK = 60              # Окно сравнения (насколько длинный текущий кусок берем, дни)
FORECAST = 40              # Прогноз (на сколько дней вперед продлеваем историю)

# --- 2. ЗАГРУЗКА ДАННЫХ ---
df = yf.download(TICKER, start="1970-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

def find_best_fractal(target_series, search_space, label):
    """Ищет один самый похожий участок в заданном пространстве данных"""
    best_score = float('inf')
    best_idx = -1

    target_norm = target_series / target_series[0]

    # Скользящее окно по истории
    for i in range(len(search_space) - LOOKBACK - FORECAST):
        sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
        sample_norm = sample / sample[0]

        # Считаем среднеквадратичную ошибку (MSE)
        score = np.mean((target_norm - sample_norm)**2)

        if score < best_score:
            best_score = score
            best_idx = i

    if best_idx != -1:
        res = search_space.iloc[best_idx : best_idx + LOOKBACK + FORECAST]['Close'].values
        date_str = search_space.index[best_idx].strftime('%Y-%m-%d')
        return (res / res[0]) * 100, date_str
    return None, None

# --- 3. ПОДГОТОВКА ПОИСКОВЫХ ЗОН ---
current_data = df['Close'].iloc[-LOOKBACK:]
four_years_ago = df.index[-1] - timedelta(days=4*365)
one_year_ago = df.index[-1] - timedelta(days=365)

# Зоны поиска
zones = {
    "Максимально возможный (вся история)": df,
    "За последние 4 года": df[df.index >= four_years_ago],
    "Годовой цикл": df[df.index >= one_year_ago]
}

# --- 4. ВИЗУАЛИЗАЦИЯ ---
fig = go.Figure()

# Текущая цена (черная жирная линия)
current_plot = (current_data.values / current_data.values[0]) * 100
fig.add_trace(go.Scatter(x=np.arange(LOOKBACK), y=current_plot,
                         name="ТЕКУЩЕЕ ДВИЖЕНИЕ", line=dict(color='black', width=5)))

colors = ['red', 'blue', 'green']
for i, (name, space) in enumerate(zones.items()):
    # Исключаем последние дни, чтобы не найти «самого себя»
    search_data = space.iloc[:-FORECAST]

    values, date_found = find_best_fractal(current_data.values, search_data, name)

    if values is not None:
        fig.add_trace(go.Scatter(
            x=np.arange(LOOKBACK + FORECAST),
            y=values,
            name=f"{name} (от {date_found})",
            line=dict(color=colors[i], width=2, dash='dot'),
            opacity=0.8
        ))

# Оформление
fig.update_layout(
    title=f"<b>Мульти-цикловой анализ {TICKER}</b><br><sup>Поиск фракталов в разных временных масштабах</sup>",
    xaxis_title="Торговые дни (от точки отсчета)",
    yaxis_title="Относительное изменение (%)",
    template="plotly_white",
    hovermode="x unified",
    shapes=[dict(type="line", x0=LOOKBACK-1, x1=LOOKBACK-1, y0=min(current_plot)*0.95, y1=max(current_plot)*1.05,
                 line=dict(color="Gray", dash="dash"))]
)

fig.show()

[*********************100%***********************]  1 of 1 completed


In [18]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime, timedelta

# --- 1. НАСТРОЙКИ ПОД КРИПТО ---
# Выберите нужный тикер: "BTC-USD" или "ETH-USD"
TICKER = "BTC-USD"
LOOKBACK = 45   # Окно анализа (1.5 месяца)
FORECAST = 30   # Прогноз на месяц вперед

# Загружаем данные (крипта торгуется ежедневно, поэтому данных больше)
df = yf.download(TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

def find_crypto_fractal(target_series, search_space):
    best_score = float('inf')
    best_idx = -1

    # Используем логарифмическое изменение для крипты (лучше передает характер движения)
    target_log = np.log(target_series)
    target_norm = (target_log - np.mean(target_log)) / np.std(target_log)

    for i in range(len(search_space) - LOOKBACK - FORECAST):
        sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
        sample_log = np.log(sample)
        sample_norm = (sample_log - np.mean(sample_log)) / np.std(sample_log)

        score = np.mean((target_norm - sample_norm)**2)
        if score < best_score:
            best_score = score
            best_idx = i

    if best_idx != -1:
        raw_values = search_space.iloc[best_idx : best_idx + LOOKBACK + FORECAST]['Close'].values
        # Совмещаем по последней цене (Price Action стыковка)
        ratio = target_series[-1] / raw_values[LOOKBACK-1]
        adjusted_values = raw_values * ratio

        corr = np.corrcoef(target_series, raw_values[:LOOKBACK])[0, 1]
        date_str = search_space.index[best_idx].strftime('%Y-%m-%d')
        return adjusted_values, date_str, corr
    return None, None, None

# --- 2. ЗАПУСК ПОИСКА ---
current_data = df['Close'].iloc[-LOOKBACK:].values
fig = go.Figure()

# Основной график BTC/ETH
fig.add_trace(go.Scatter(y=current_data, name=f"ТЕКУЩИЙ {TICKER}",
                         line=dict(color='#F7931A' if "BTC" in TICKER else '#627EEA', width=4)))

# Поиск зон
search_zones = [
    ("Глобальный цикл", df, 'red'),
    ("Цикл после Халвинга (последние 4г)", df[df.index >= df.index[-1] - timedelta(days=4*365)], 'blue'),
    ("Краткосрочный тренд (1г)", df[df.index >= df.index[-1] - timedelta(days=365)], 'green')
]

for name, space, color in search_zones:
    # Отрезаем хвост, чтобы не сравнивать с самим собой
    vals, date_found, correlation = find_crypto_fractal(current_data, space.iloc[:-FORECAST])
    if vals is not None:
        fig.add_trace(go.Scatter(
            y=vals,
            name=f"{name} ({date_found})<br>Сходство: {correlation:.2%}",
            line=dict(color=color, width=2, dash='dot'),
            opacity=0.6
        ))

# --- 3. ОФОРМЛЕНИЕ ---
fig.update_layout(
    title=f"<b>Фрактальный прогноз для {TICKER}</b>",
    yaxis_type="log", # ВКЛЮЧАЕМ ЛОГАРИФМИЧЕСКУЮ ШКАЛУ
    xaxis_title="Дни", yaxis_title="Цена (Log Scale)",
    hovermode="x unified", template="plotly_white",
    shapes=[dict(type="line", x0=LOOKBACK-1, x1=LOOKBACK-1,
                 y0=min(current_data)*0.8, y1=max(current_data)*1.2,
                 line=dict(color="gray", dash="dash"))]
)
fig.show()

[*********************100%***********************]  1 of 1 completed


In [17]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime, timedelta

# --- 1. НАСТРОЙКИ ПОД КРИПТО ---
# Выберите нужный тикер: "BTC-USD" или "ETH-USD"
TICKER = "ETH-USD"
LOOKBACK = 45   # Окно анализа (1.5 месяца)
FORECAST = 30   # Прогноз на месяц вперед

# Загружаем данные (крипта торгуется ежедневно, поэтому данных больше)
df = yf.download(TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

def find_crypto_fractal(target_series, search_space):
    best_score = float('inf')
    best_idx = -1

    # Используем логарифмическое изменение для крипты (лучше передает характер движения)
    target_log = np.log(target_series)
    target_norm = (target_log - np.mean(target_log)) / np.std(target_log)

    for i in range(len(search_space) - LOOKBACK - FORECAST):
        sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
        sample_log = np.log(sample)
        sample_norm = (sample_log - np.mean(sample_log)) / np.std(sample_log)

        score = np.mean((target_norm - sample_norm)**2)
        if score < best_score:
            best_score = score
            best_idx = i

    if best_idx != -1:
        raw_values = search_space.iloc[best_idx : best_idx + LOOKBACK + FORECAST]['Close'].values
        # Совмещаем по последней цене (Price Action стыковка)
        ratio = target_series[-1] / raw_values[LOOKBACK-1]
        adjusted_values = raw_values * ratio

        corr = np.corrcoef(target_series, raw_values[:LOOKBACK])[0, 1]
        date_str = search_space.index[best_idx].strftime('%Y-%m-%d')
        return adjusted_values, date_str, corr
    return None, None, None

# --- 2. ЗАПУСК ПОИСКА ---
current_data = df['Close'].iloc[-LOOKBACK:].values
fig = go.Figure()

# Основной график BTC/ETH
fig.add_trace(go.Scatter(y=current_data, name=f"ТЕКУЩИЙ {TICKER}",
                         line=dict(color='#F7931A' if "BTC" in TICKER else '#627EEA', width=4)))

# Поиск зон
search_zones = [
    ("Глобальный цикл", df, 'red'),
    ("Цикл после Халвинга (последние 4г)", df[df.index >= df.index[-1] - timedelta(days=4*365)], 'blue'),
    ("Краткосрочный тренд (1г)", df[df.index >= df.index[-1] - timedelta(days=365)], 'green')
]

for name, space, color in search_zones:
    # Отрезаем хвост, чтобы не сравнивать с самим собой
    vals, date_found, correlation = find_crypto_fractal(current_data, space.iloc[:-FORECAST])
    if vals is not None:
        fig.add_trace(go.Scatter(
            y=vals,
            name=f"{name} ({date_found})<br>Сходство: {correlation:.2%}",
            line=dict(color=color, width=2, dash='dot'),
            opacity=0.6
        ))

# --- 3. ОФОРМЛЕНИЕ ---
fig.update_layout(
    title=f"<b>Фрактальный прогноз для {TICKER}</b>",
    yaxis_type="log", # ВКЛЮЧАЕМ ЛОГАРИФМИЧЕСКУЮ ШКАЛУ
    xaxis_title="Дни", yaxis_title="Цена (Log Scale)",
    hovermode="x unified", template="plotly_white",
    shapes=[dict(type="line", x0=LOOKBACK-1, x1=LOOKBACK-1,
                 y0=min(current_data)*0.8, y1=max(current_data)*1.2,
                 line=dict(color="gray", dash="dash"))]
)
fig.show()

[*********************100%***********************]  1 of 1 completed


In [16]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- НАСТРОЙКИ ---
TICKER = "BTC-USD"  # Можно сменить на ETH-USD или ^GSPC
LOOKBACK = 60       # Сколько дней анализируем (прошлое)
FORECAST = 60       # На сколько дней смотрим вперед (будущее)

# Загрузка данных
df = yf.download(TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

def get_forecast_with_dates(target_series, search_space, current_dates):
    best_score = float('inf')
    best_idx = -1

    # Нормализация для поиска формы
    target_norm = (target_series - np.mean(target_series)) / np.std(target_series)

    for i in range(len(search_space) - LOOKBACK - FORECAST):
        sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
        sample_norm = (sample - np.mean(sample)) / np.std(sample)
        score = np.mean((target_norm - sample_norm)**2)

        if score < best_score:
            best_score = score
            best_idx = i

    if best_idx != -1:
        # Извлекаем исторические значения (включая будущий период)
        raw_values = search_space.iloc[best_idx : best_idx + LOOKBACK + FORECAST]['Close'].values
        # Стыковка по последней цене текущего графика
        ratio = target_series[-1] / raw_values[LOOKBACK-1]
        forecast_values = raw_values * ratio

        # Создаем будущую шкалу времени
        last_date = current_dates[-1]
        future_dates = [last_date + timedelta(days=i) for i in range(1, FORECAST + 1)]
        full_dates = list(current_dates) + future_dates

        return full_dates, forecast_values, search_space.index[best_idx].strftime('%Y-%m-%d')
    return None, None, None

# --- ПОДГОТОВКА ДАННЫХ ---
current_df = df.iloc[-LOOKBACK:]
current_prices = current_df['Close'].values
current_dates = current_df.index

fig = go.Figure()

# 1. Реальный график (черный)
fig.add_trace(go.Scatter(x=current_dates, y=current_prices, name="ТЕКУЩАЯ ЦЕНА",
                         line=dict(color='black', width=4)))

# 2. Поиск и отрисовка циклов
colors = {'Глобальный': 'red', '4 года': 'blue', 'Годовой': 'green'}
zones = [
    ("Глобальный", df),
    ("4 года", df[df.index >= df.index[-1] - timedelta(days=4*365)]),
    ("Годовой", df[df.index >= df.index[-1] - timedelta(days=365)])
]

for name, space in zones:
    dates, values, hist_start = get_forecast_with_dates(current_prices, space.iloc[:-FORECAST], current_dates)

    if values is not None:
        # Рисуем линию цикла
        fig.add_trace(go.Scatter(x=dates, y=values, name=f"Цикл: {name} ({hist_start})",
                                 line=dict(color=colors[name], width=2, dash='dot'), opacity=0.5))

        # Находим экстремумы в ПРЕДСКАЗАННОЙ части (после текущей даты)
        forecast_part = values[LOOKBACK:]
        forecast_dates = dates[LOOKBACK:]

        max_idx = np.argmax(forecast_part)
        min_idx = np.argmin(forecast_part)

        # Добавляем метку ПИКА
        fig.add_annotation(x=forecast_dates[max_idx], y=forecast_part[max_idx],
                           text=f"Пик {forecast_dates[max_idx].strftime('%d.%m')}",
                           showarrow=True, arrowhead=1, bgcolor=colors[name], font=dict(color="white"))

        # Добавляем метку ДНА
        fig.add_annotation(x=forecast_dates[min_idx], y=forecast_part[min_idx],
                           text=f"Дно {forecast_dates[min_idx].strftime('%d.%m')}",
                           showarrow=True, arrowhead=1, bgcolor="black", font=dict(color="white"), ay=40)

# --- ОФОРМЛЕНИЕ ---
fig.update_layout(
    title=f"<b>Прогноз по датам: {TICKER}</b>",
    xaxis_title="Календарная дата",
    yaxis_title="Цена",
    template="plotly_white",
    hovermode="x unified",
    # Вертикальная линия "СЕГОДНЯ"
    shapes=[dict(type="line", x0=current_dates[-1], x1=current_dates[-1],
                 y0=min(current_prices)*0.8, y1=max(current_prices)*1.2,
                 line=dict(color="Gray", width=2, dash="dash"))]
)

# Переключаем на логарифмическую шкалу для крипты
if "USD" in TICKER: fig.update_yaxes(type="log")

fig.show()

[*********************100%***********************]  1 of 1 completed


In [15]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- НАСТРОЙКИ ---
TICKER = "ETH-USD"  # Можно сменить на ETH-USD или ^GSPC
LOOKBACK = 60       # Сколько дней анализируем (прошлое)
FORECAST = 60       # На сколько дней смотрим вперед (будущее)

# Загрузка данных
df = yf.download(TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

def get_forecast_with_dates(target_series, search_space, current_dates):
    best_score = float('inf')
    best_idx = -1

    # Нормализация для поиска формы
    target_norm = (target_series - np.mean(target_series)) / np.std(target_series)

    for i in range(len(search_space) - LOOKBACK - FORECAST):
        sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
        sample_norm = (sample - np.mean(sample)) / np.std(sample)
        score = np.mean((target_norm - sample_norm)**2)

        if score < best_score:
            best_score = score
            best_idx = i

    if best_idx != -1:
        # Извлекаем исторические значения (включая будущий период)
        raw_values = search_space.iloc[best_idx : best_idx + LOOKBACK + FORECAST]['Close'].values
        # Стыковка по последней цене текущего графика
        ratio = target_series[-1] / raw_values[LOOKBACK-1]
        forecast_values = raw_values * ratio

        # Создаем будущую шкалу времени
        last_date = current_dates[-1]
        future_dates = [last_date + timedelta(days=i) for i in range(1, FORECAST + 1)]
        full_dates = list(current_dates) + future_dates

        return full_dates, forecast_values, search_space.index[best_idx].strftime('%Y-%m-%d')
    return None, None, None

# --- ПОДГОТОВКА ДАННЫХ ---
current_df = df.iloc[-LOOKBACK:]
current_prices = current_df['Close'].values
current_dates = current_df.index

fig = go.Figure()

# 1. Реальный график (черный)
fig.add_trace(go.Scatter(x=current_dates, y=current_prices, name="ТЕКУЩАЯ ЦЕНА",
                         line=dict(color='black', width=4)))

# 2. Поиск и отрисовка циклов
colors = {'Глобальный': 'red', '4 года': 'blue', 'Годовой': 'green'}
zones = [
    ("Глобальный", df),
    ("4 года", df[df.index >= df.index[-1] - timedelta(days=4*365)]),
    ("Годовой", df[df.index >= df.index[-1] - timedelta(days=365)])
]

for name, space in zones:
    dates, values, hist_start = get_forecast_with_dates(current_prices, space.iloc[:-FORECAST], current_dates)

    if values is not None:
        # Рисуем линию цикла
        fig.add_trace(go.Scatter(x=dates, y=values, name=f"Цикл: {name} ({hist_start})",
                                 line=dict(color=colors[name], width=2, dash='dot'), opacity=0.5))

        # Находим экстремумы в ПРЕДСКАЗАННОЙ части (после текущей даты)
        forecast_part = values[LOOKBACK:]
        forecast_dates = dates[LOOKBACK:]

        max_idx = np.argmax(forecast_part)
        min_idx = np.argmin(forecast_part)

        # Добавляем метку ПИКА
        fig.add_annotation(x=forecast_dates[max_idx], y=forecast_part[max_idx],
                           text=f"Пик {forecast_dates[max_idx].strftime('%d.%m')}",
                           showarrow=True, arrowhead=1, bgcolor=colors[name], font=dict(color="white"))

        # Добавляем метку ДНА
        fig.add_annotation(x=forecast_dates[min_idx], y=forecast_part[min_idx],
                           text=f"Дно {forecast_dates[min_idx].strftime('%d.%m')}",
                           showarrow=True, arrowhead=1, bgcolor="black", font=dict(color="white"), ay=40)

# --- ОФОРМЛЕНИЕ ---
fig.update_layout(
    title=f"<b>Прогноз по датам: {TICKER}</b>",
    xaxis_title="Календарная дата",
    yaxis_title="Цена",
    template="plotly_white",
    hovermode="x unified",
    # Вертикальная линия "СЕГОДНЯ"
    shapes=[dict(type="line", x0=current_dates[-1], x1=current_dates[-1],
                 y0=min(current_prices)*0.8, y1=max(current_prices)*1.2,
                 line=dict(color="Gray", width=2, dash="dash"))]
)

# Переключаем на логарифмическую шкалу для крипты
if "USD" in TICKER: fig.update_yaxes(type="log")

fig.show()

[*********************100%***********************]  1 of 1 completed


In [14]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- НАСТРОЙКИ ---
TICKER = "BTC-USD"
LOOKBACK = 60       # Дни для анализа
FORECAST = 45       # Прогноз
TOP_N = 3           # Количество лучших циклов

# Загрузка данных
df = yf.download(TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

current_df = df.iloc[-LOOKBACK:]
current_prices = current_df['Close'].values
current_dates = current_df.index

# Нормализация текущего движения (Z-score)
target_norm = (current_prices - np.mean(current_prices)) / np.std(current_prices)

results = []

# Поиск всех возможных циклов
search_space = df.iloc[:-FORECAST]
for i in range(len(search_space) - LOOKBACK):
    sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
    sample_norm = (sample - np.mean(sample)) / np.std(sample)

    # Считаем MSE
    score = np.mean((target_norm - sample_norm)**2)
    results.append((score, i))

# Сортируем по схожести и берем топ-3
# Фильтруем, чтобы циклы не накладывались друг на друга слишком сильно (минимум 30 дней разницы)
results.sort(key=lambda x: x[0])
best_cycles = []
used_indices = []

for score, idx in results:
    if not any(abs(idx - used_idx) < 30 for used_idx in used_indices):
        best_cycles.append((score, idx))
        used_indices.append(idx)
    if len(best_cycles) == TOP_N: break

# --- ВИЗУАЛИЗАЦИЯ ---
fig = go.Figure()

# Текущая цена
fig.add_trace(go.Scatter(x=np.arange(LOOKBACK), y=current_prices,
                         name="ТЕКУЩАЯ ЦЕНА", line=dict(color='black', width=4)))

colors = ['red', 'blue', 'green']
for i, (score, idx) in enumerate(best_cycles):
    raw_values = search_space.iloc[idx : idx + LOOKBACK + FORECAST]['Close'].values
    # Стыковка по последней цене
    ratio = current_prices[-1] / raw_values[LOOKBACK-1]
    forecast_values = raw_values * ratio

    start_date = search_space.index[idx].strftime('%Y-%m-%d')
    fig.add_trace(go.Scatter(
        y=forecast_values,
        name=f"#{i+1} Цикл ({start_date})",
        line=dict(color=colors[i], width=2, dash='dot'),
        opacity=0.7
    ))

fig.update_layout(
    title=f"<b>Топ-3 похожих цикла для {TICKER}</b>",
    xaxis_title="Дни", yaxis_title="Цена",
    template="plotly_white", hovermode="x unified",
    shapes=[dict(type="line", x0=LOOKBACK-1, x1=LOOKBACK-1, y0=min(current_prices)*0.9, y1=max(current_prices)*1.1, line=dict(color="gray", dash="dash"))]
)
if "USD" in TICKER: fig.update_yaxes(type="log")
fig.show()

[*********************100%***********************]  1 of 1 completed


In [20]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- 1. НАСТРОЙКИ ---
TICKER = "BTC-USD"  # Можно заменить на ETH-USD
LOOKBACK = 90       # Анализируем последние 3 месяца
FORECAST = 30       # Прогноз на 1 месяц

# Загрузка данных
df = yf.download(TICKER, start="2010-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

# Текущее движение
current_prices = df['Close'].iloc[-LOOKBACK:].values
current_dates = df.index[-LOOKBACK:]

# --- 2. ПОИСК АБСОЛЮТНО ЛУЧШЕГО ЦИКЛА ---
best_corr = -1
best_idx = -1

# Проходим по всей истории (исключая будущий период прогноза)
search_space = df.iloc[:-FORECAST]
for i in range(len(search_space) - LOOKBACK):
    sample = search_space.iloc[i : i + LOOKBACK]['Close'].values

    # Считаем корреляцию Пирсона (насколько формы идентичны)
    corr = np.corrcoef(current_prices, sample)[0, 1]

    if corr > best_corr:
        best_corr = corr
        best_idx = i

# --- 3. ПОДГОТОВКА И ВИЗУАЛИЗАЦИЯ ---
best_cycle_data = df.iloc[best_idx : best_idx + LOOKBACK + FORECAST]
best_date = best_cycle_data.index[0].strftime('%Y-%m-%d')

# Стыкуем исторические данные с текущей ценой
ratio = current_prices[-1] / best_cycle_data['Close'].values[LOOKBACK-1]
projected_prices = best_cycle_data['Close'].values * ratio

# Создаем шкалу времени для прогноза
future_dates = [current_dates[-1] + timedelta(days=i) for i in range(1, FORECAST + 1)]
all_dates = list(current_dates) + future_dates

fig = go.Figure()

# Текущая цена
fig.add_trace(go.Scatter(x=current_dates, y=current_prices, name="Текущая цена", line=dict(color='black', width=4)))

# Лучший найденный цикл
fig.add_trace(go.Scatter(x=all_dates, y=projected_prices, name=f"Лучший цикл ({best_date})",
                         line=dict(color='red', width=2, dash='dot')))

# Индикатор прогноза
fig.add_vrect(x0=current_dates[-1], x1=all_dates[-1], fillcolor="green", opacity=0.1,
              layer="below", line_width=0, annotation_text="ПРОГНОЗ")

fig.update_layout(
    title=f"<b>Лучший исторический цикл для {TICKER}</b><br><sup>Сходство (Correlation): {best_corr:.2%} | Начало цикла: {best_date}</sup>",
    xaxis_title="Дата", yaxis_title="Цена",
    template="plotly_white", hovermode="x unified"
)

if "USD" in TICKER: fig.update_yaxes(type="log")
fig.show()

[*********************100%***********************]  1 of 1 completed


In [21]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- 1. НАСТРОЙКИ ---
TICKER = "ETH-USD"  # Можно заменить на ETH-USD
LOOKBACK = 90       # Анализируем последние 3 месяца
FORECAST = 30       # Прогноз на 1 месяц

# Загрузка данных
df = yf.download(TICKER, start="2010-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

# Текущее движение
current_prices = df['Close'].iloc[-LOOKBACK:].values
current_dates = df.index[-LOOKBACK:]

# --- 2. ПОИСК АБСОЛЮТНО ЛУЧШЕГО ЦИКЛА ---
best_corr = -1
best_idx = -1

# Проходим по всей истории (исключая будущий период прогноза)
search_space = df.iloc[:-FORECAST]
for i in range(len(search_space) - LOOKBACK):
    sample = search_space.iloc[i : i + LOOKBACK]['Close'].values

    # Считаем корреляцию Пирсона (насколько формы идентичны)
    corr = np.corrcoef(current_prices, sample)[0, 1]

    if corr > best_corr:
        best_corr = corr
        best_idx = i

# --- 3. ПОДГОТОВКА И ВИЗУАЛИЗАЦИЯ ---
best_cycle_data = df.iloc[best_idx : best_idx + LOOKBACK + FORECAST]
best_date = best_cycle_data.index[0].strftime('%Y-%m-%d')

# Стыкуем исторические данные с текущей ценой
ratio = current_prices[-1] / best_cycle_data['Close'].values[LOOKBACK-1]
projected_prices = best_cycle_data['Close'].values * ratio

# Создаем шкалу времени для прогноза
future_dates = [current_dates[-1] + timedelta(days=i) for i in range(1, FORECAST + 1)]
all_dates = list(current_dates) + future_dates

fig = go.Figure()

# Текущая цена
fig.add_trace(go.Scatter(x=current_dates, y=current_prices, name="Текущая цена", line=dict(color='black', width=4)))

# Лучший найденный цикл
fig.add_trace(go.Scatter(x=all_dates, y=projected_prices, name=f"Лучший цикл ({best_date})",
                         line=dict(color='red', width=2, dash='dot')))

# Индикатор прогноза
fig.add_vrect(x0=current_dates[-1], x1=all_dates[-1], fillcolor="green", opacity=0.1,
              layer="below", line_width=0, annotation_text="ПРОГНОЗ")

fig.update_layout(
    title=f"<b>Лучший исторический цикл для {TICKER}</b><br><sup>Сходство (Correlation): {best_corr:.2%} | Начало цикла: {best_date}</sup>",
    xaxis_title="Дата", yaxis_title="Цена",
    template="plotly_white", hovermode="x unified"
)

if "USD" in TICKER: fig.update_yaxes(type="log")
fig.show()

[*********************100%***********************]  1 of 1 completed


In [22]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- 1. НАСТРОЙКИ ---
TICKER = "ETH-USD"
LOOKBACK_RANGE = range(30, 121, 5) # Ищем оптимальное окно от 30 до 120 дней
FORECAST = 30

# Загрузка данных
df = yf.download(TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

# --- 2. ГЛУБИННЫЙ АВТОПОДБОР (GRID SEARCH) ---
overall_best_corr = -1
overall_best_idx = -1
overall_best_lb = -1

print("Запуск глубинного поиска лучшего окна...")

for lb in LOOKBACK_RANGE:
    current_prices = df['Close'].iloc[-lb:].values
    search_space = df.iloc[:-FORECAST - lb]

    # Поиск лучшего совпадения для данного окна
    for i in range(len(search_space) - lb):
        sample = search_space.iloc[i : i + lb]['Close'].values
        corr = np.corrcoef(current_prices, sample)[0, 1]

        if corr > overall_best_corr:
            overall_best_corr = corr
            overall_best_idx = i
            overall_best_lb = lb

print(f"\nОптимальное окно (LOOKBACK) найдено: {overall_best_lb} дней")
print(f"Максимальная корреляция: {overall_best_corr:.2%}")

# --- 3. ВИЗУАЛИЗАЦИЯ ЛУЧШЕГО ИЗ ЛУЧШИХ ---
best_cycle_data = df.iloc[overall_best_idx : overall_best_idx + overall_best_lb + FORECAST]
best_date = best_cycle_data.index[0].strftime('%Y-%m-%d')
current_prices_final = df['Close'].iloc[-overall_best_lb:].values
current_dates_final = df.index[-overall_best_lb:]

# Стыковка
ratio = current_prices_final[-1] / best_cycle_data['Close'].values[overall_best_lb-1]
projected_prices = best_cycle_data['Close'].values * ratio

future_dates = [current_dates_final[-1] + timedelta(days=i) for i in range(1, FORECAST + 1)]
all_dates = list(current_dates_final) + future_dates

fig = go.Figure()
fig.add_trace(go.Scatter(x=current_dates_final, y=current_prices_final, name="Текущая цена", line=dict(color='black', width=4)))
fig.add_trace(go.Scatter(x=all_dates, y=projected_prices, name=f"Лучший фрактал ({best_date}, LB: {overall_best_lb})", line=dict(color='red', width=2, dash='dot')))

fig.add_vrect(x0=current_dates_final[-1], x1=all_dates[-1], fillcolor="blue", opacity=0.05, annotation_text="ЗОНА ПРОГНОЗА")

fig.update_layout(
    title=f"<b>Глубинный автоподбор фрактала для {TICKER}</b><br><sup>Лучшее окно: {overall_best_lb} дн. | Корреляция: {overall_best_corr:.2%}</sup>",
    template="plotly_white", hovermode="x unified"
)
if "USD" in TICKER: fig.update_yaxes(type="log")
fig.show()

[*********************100%***********************]  1 of 1 completed


Запуск глубинного поиска лучшего окна...

Оптимальное окно (LOOKBACK) найдено: 45 дней
Максимальная корреляция: 91.44%


In [23]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- FINAL CONSOLIDATED PARAMETERS ---
TICKER = "ETH-USD"
OPTIMIZED_LB = 45 # Derived from previous Grid Search
FORECAST = 45

# Load fresh data
df = yf.download(TICKER, start="2014-01-01", auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

def find_fractal(target_prices, search_space, lookback, forecast, min_corr=0.0):
    best_corr = -1
    best_idx = -1
    for i in range(len(search_space) - lookback - forecast):
        sample = search_space.iloc[i : i + lookback]['Close'].values
        corr = np.corrcoef(target_prices, sample)[0, 1]
        if corr > best_corr:
            best_corr = corr
            best_idx = i

    if best_idx != -1 and best_corr >= min_corr:
        raw = search_space.iloc[best_idx : best_idx + lookback + forecast]['Close'].values
        ratio = target_prices[-1] / raw[lookback-1]
        return raw * ratio, best_corr, search_space.index[best_idx].strftime('%Y-%m-%d')
    return None, None, None

# Prepare target series
target_opt = df['Close'].iloc[-OPTIMIZED_LB:].values
target_std = df['Close'].iloc[-60:].values # Standard 60d for other comparisons

fig = go.Figure()

# 1. ACTUAL PRICE
fig.add_trace(go.Scatter(x=np.arange(60), y=target_std, name="CURRENT PRICE", line=dict(color='black', width=5)))

# CONFIGURATIONS FOR MULTI-FORECAST
scenarios = [
    ("Optimized (45d)", df, OPTIMIZED_LB, 0.0, 'red'),
    ("Global (60d)", df, 60, 0.0, 'blue'),
    ("Halving (4yr)", df[df.index >= df.index[-1] - timedelta(days=4*365)], 60, 0.0, 'green'),
    ("Annual (1yr)", df[df.index >= df.index[-1] - timedelta(days=365)], 60, 0.0, 'orange'),
    ("High Confidence (>90%)", df, 60, 0.90, 'purple')
]

for name, space, lb, min_c, color in scenarios:
    search_data = space.iloc[:-FORECAST]
    current_target = df['Close'].iloc[-lb:].values
    vals, corr, d_start = find_fractal(current_target, search_data, lb, FORECAST, min_c)

    if vals is not None:
        # Align x-axis so 0 is start of lookback relative to current 60d window
        x_offset = 60 - lb
        fig.add_trace(go.Scatter(
            x=np.arange(x_offset, x_offset + lb + FORECAST),
            y=vals,
            name=f"{name} (Corr: {corr:.1%})",
            line=dict(color=color, width=2, dash='dot'),
            opacity=0.7
        ))

fig.add_vrect(x0=59, x1=59+FORECAST, fillcolor="gray", opacity=0.1, annotation_text="FORECAST")
fig.update_layout(
    title=f"<b>Final Consolidated Fractal Research: {TICKER}</b>",
    xaxis_title="Relative Days", yaxis_title="Price (USD)",
    yaxis_type="log", template="plotly_white", hovermode="x unified"
)
fig.show()

[*********************100%***********************]  1 of 1 completed


In [26]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- ПАРАМЕТРЫ КОНСОЛИДАЦИИ ---
TICKER = "ETH-USD"
LOOKBACK_OPT = 45
LOOKBACK_STD = 60
FORECAST = 45

df = yf.download(TICKER, start='2014-01-01', auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

def get_fractal(target, search_space, lb, fc, method='mse'):
    best_score = float('inf') if method == 'mse' else -1
    best_idx = -1

    t_norm = (target - np.mean(target)) / np.std(target)

    for i in range(len(search_space) - lb - fc):
        sample = search_space.iloc[i : i + lb]['Close'].values
        s_norm = (sample - np.mean(sample)) / np.std(sample)

        if method == 'mse':
            score = np.mean((t_norm - s_norm)**2)
            if score < best_score:
                best_score, best_idx = score, i
        else: # correlation
            score = np.corrcoef(target, sample)[0, 1]
            if score > best_score:
                best_score, best_idx = score, i

    if best_idx != -1:
        raw = search_space.iloc[best_idx : best_idx + lb + fc]['Close'].values
        ratio = target[-1] / raw[lb-1]
        return raw * ratio, best_score, search_space.index[best_idx]
    return None, None, None

# Подготовка данных
current_prices = df['Close'].iloc[-LOOKBACK_STD:].values
search_area = df.iloc[:-FORECAST]

fig = go.Figure()

# 1. ТЕКУЩАЯ ЦЕНА
fig.add_trace(go.Scatter(x=np.arange(LOOKBACK_STD), y=current_prices, name="ФАКТ (ETH)", line=dict(color='black', width=5)))

# Сценарии
scenarios = [
    ('Оптимальное окно (45д)', LOOKBACK_OPT, 'mse', 'red'),
    ('Зигзаг/MSE (60д)', LOOKBACK_STD, 'mse', 'blue'),
    ('Корреляция (60д)', LOOKBACK_STD, 'corr', 'green'),
    ('Глобальный поиск', 90, 'mse', 'orange'),
    ('High Confidence (>90%)', 60, 'corr', 'purple')
]

for name, lb, method, color in scenarios:
    target = df['Close'].iloc[-lb:].values
    vals, score, d_start = get_fractal(target, search_area, lb, FORECAST, method)

    if vals is not None:
        # Если это High Confidence, проверяем порог
        if 'Confidence' in name and score < 0.90: continue

        offset = LOOKBACK_STD - lb
        label = f"{name} ({score:.2f})" if method == 'mse' else f"{name} ({score:.1%})"

        fig.add_trace(go.Scatter(
            x=np.arange(offset, offset + lb + FORECAST),
            y=vals,
            name=label,
            line=dict(color=color, width=2, dash='dot'),
            opacity=0.6
        ))

fig.add_vrect(x0=LOOKBACK_STD-1, x1=LOOKBACK_STD+FORECAST-1, fillcolor="gray", opacity=0.1, annotation_text="ЗОНА ПРОГНОЗА")
fig.update_layout(
    title=f"<b>Финальный консолидированный прогноз {TICKER}</b>",
    xaxis_title="Относительные дни", yaxis_title="Цена (Log)",
    yaxis_type="log", template="plotly_white", hovermode="x unified"
)
fig.show()

[*********************100%***********************]  1 of 1 completed


# FINAL CONSOLIDATED RESEARCH
This block synthesizes the three primary forecasting methods: Optimized Lookback, Global Historical Search, and High-Confidence Correlation matching.

In [27]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- SETTINGS ---
TICKER = "ETH-USD"
OPTIMIZED_LB = 45
STD_LB = 60
FORECAST = 45

df = yf.download(TICKER, start='2014-01-01', auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

def find_fractal_logic(target, search_space, lb, fc, mode='mse'):
    best_val = float('inf') if mode == 'mse' else -1
    best_idx = -1
    t_norm = (target - np.mean(target)) / np.std(target)

    for i in range(len(search_space) - lb - fc):
        sample = search_space.iloc[i : i + lb]['Close'].values
        if mode == 'mse':
            s_norm = (sample - np.mean(sample)) / np.std(sample)
            score = np.mean((t_norm - s_norm)**2)
            if score < best_val: best_val, best_idx = score, i
        else:
            score = np.corrcoef(target, sample)[0, 1]
            if score > best_val: best_val, best_idx = score, i

    if best_idx != -1:
        raw = search_space.iloc[best_idx : best_idx + lb + fc]['Close'].values
        ratio = target[-1] / raw[lb-1]
        return raw * ratio, best_val, search_space.index[best_idx]
    return None, None, None

# Data Prep
current_prices = df['Close'].iloc[-STD_LB:].values
search_area = df.iloc[:-FORECAST]
fig = go.Figure()

# 1. ACTUAL PRICE
fig.add_trace(go.Scatter(x=np.arange(STD_LB), y=current_prices, name="ACTUAL ETH", line=dict(color='black', width=5)))

# 2. SCENARIOS
# A: Optimized Window (45d)
target_opt = df['Close'].iloc[-OPTIMIZED_LB:].values
vals_opt, score_opt, date_opt = find_fractal_logic(target_opt, search_area, OPTIMIZED_LB, FORECAST, 'mse')
if vals_opt is not None:
    fig.add_trace(go.Scatter(x=np.arange(STD_LB-OPTIMIZED_LB, STD_LB+FORECAST), y=vals_opt,
                             name=f"Optimized (LB:45, MSE:{score_opt:.2f})", line=dict(color='red', width=2, dash='dot')))

# B: Global Search (Standard 60d)
vals_gb, score_gb, date_gb = find_fractal_logic(current_prices, search_area, STD_LB, FORECAST, 'mse')
if vals_gb is not None:
    fig.add_trace(go.Scatter(x=np.arange(STD_LB+FORECAST), y=vals_gb,
                             name=f"Global (LB:60, MSE:{score_gb:.2f})", line=dict(color='orange', width=2, dash='dot')))

# C: High Confidence Correlation (>90%)
vals_hc, score_hc, date_hc = find_fractal_logic(current_prices, search_area, STD_LB, FORECAST, 'corr')
if vals_hc is not None and score_hc >= 0.90:
    fig.add_trace(go.Scatter(x=np.arange(STD_LB+FORECAST), y=vals_hc,
                             name=f"High Confidence ({score_hc:.1%})", line=dict(color='purple', width=3)))

# Layout
fig.add_vrect(x0=STD_LB-1, x1=STD_LB+FORECAST-1, fillcolor="gray", opacity=0.1, annotation_text="FORECAST ZONE")
fig.update_layout(title=f"<b>Final Consolidated Forecast: {TICKER}</b>", yaxis_type="log",
                  xaxis_title="Relative Days", yaxis_title="Price (USD)", template="plotly_white", hovermode="x unified")
fig.show()

[*********************100%***********************]  1 of 1 completed


In [28]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import timedelta

# --- SETTINGS ---
TICKER = "ETH-USD"
LOOKBACK = 45
FORECAST = 45
TOP_N = 3

# Load data
df = yf.download(TICKER, start='2014-01-01', auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

# Current pattern
target = df['Close'].iloc[-LOOKBACK:].values
target_norm = (target - np.mean(target)) / np.std(target)

# Search all history
search_space = df.iloc[:-FORECAST]
results = []

for i in range(len(search_space) - LOOKBACK):
    sample = search_space.iloc[i : i + LOOKBACK]['Close'].values
    sample_norm = (sample - np.mean(sample)) / np.std(sample)

    mse = np.mean((target_norm - sample_norm)**2)
    results.append((mse, i))

# Filter for distinct non-overlapping cycles (min 30 days apart)
results.sort(key=lambda x: x[0])
best_matches = []
used_indices = []

for mse, idx in results:
    if not any(abs(idx - u) < 30 for u in used_indices):
        best_matches.append((mse, idx))
        used_indices.append(idx)
    if len(best_matches) == TOP_N: break

# Visualization
fig = go.Figure()

# Actual Price
fig.add_trace(go.Scatter(x=np.arange(LOOKBACK), y=target, name="CURRENT ETH", line=dict(color='black', width=5)))

colors = ['red', 'blue', 'green']
for i, (mse, idx) in enumerate(best_matches):
    raw = df.iloc[idx : idx + LOOKBACK + FORECAST]['Close'].values
    ratio = target[-1] / raw[LOOKBACK-1]
    vals = raw * ratio
    start_date = df.index[idx].strftime('%Y-%m-%d')

    fig.add_trace(go.Scatter(
        x=np.arange(LOOKBACK + FORECAST),
        y=vals,
        name=f"Match #{i+1} ({start_date}) MSE: {mse:.3f}",
        line=dict(color=colors[i], width=2, dash='dot'),
        opacity=0.7
    ))

fig.add_vrect(x0=LOOKBACK-1, x1=LOOKBACK+FORECAST-1, fillcolor="gray", opacity=0.1, annotation_text="PROJECTION")
fig.update_layout(
    title=f"<b>Top {TOP_N} Best Historical Matches for {TICKER} (Lowest MSE)</b>",
    yaxis_type="log",
    xaxis_title="Relative Days",
    yaxis_title="Price (USD)",
    template="plotly_white",
    hovermode="x unified"
)
fig.show()

[*********************100%***********************]  1 of 1 completed
